# Data Ingestion & Churn Definition
**Purpose:** Load the billings table, apply the three-rule churn definition, and produce the clean `model_df` used by all downstream notebooks.

In [0]:
%run ./NB01_Config_and_Setup

## 1 · Load Billings Table

In [0]:
section("Loading billings table")
billings = load_table(BILLINGS_TABLE, BILLINGS_PATH)
display_df(billings, "Billings — first 5 rows", n=5)
print("\nColumn list:")
print(list(billings.columns))


## 2 · Parse Dates & Core Columns

In [0]:
DATE_COLS = ['Prospect_Renewal_Date', 'Closed_Date', 'Registration_Date',
             'Proforma_Date', 'Last_Years_Date_Paid']
billings = parse_dates(billings, DATE_COLS)

# Numeric coercions
NUM_COLS = ['Auto_Renewal_Score','Status_Scores','Anchoring_Score','Tenure_Scores',
            'Tenure_Years','Current_Anchorings','Last_Years_Price','Total_Renewal_Score_New',
            'Starting_Net','Sustainability_Score','Renewal_Score_At_Release',
            'Total_Net_Paid','Gross','Amount','Total_Amount']
for c in NUM_COLS:
    if c in billings.columns:
        billings[c] = pd.to_numeric(billings[c], errors='coerce')

print("[NB02] Date and numeric columns parsed")
print(f"  Closed_Date null count     : {billings['Closed_Date'].isna().sum():,}")


## 3 · Churn Definition — Three Rules

| Rule | Condition | Label |
|------|-----------|-------|
| **True Churn** | `Prospect_Outcome = 'Churned'` AND `Closed_Date is NULL` | Churned |
| **True Churn** | `Prospect_Outcome = 'Churned'` AND `(Closed_Date - Prospect_Renewal_Date) > 28 days` | Churned |
| **Pre-Churned** | `Prospect_Outcome = 'Churned'` AND `Closed_Date < Prospect_Renewal_Date` | Pre_Churned (excluded) |
| **Won** | `Prospect_Outcome = 'Won'` | Won |
| **Open** | `Prospect_Outcome = 'Open'` | Open (excluded from model) |

> **Why 28 days?** A member who closes more than 28 days after their renewal date has effectively lapsed — late closure is operationally equivalent to non-renewal. Members who close within 28 days are counted as retained (Won).

> **Why exclude Pre-Churned?** These members left *before* their renewal window opened. The cause is early cancellation or business closure — a fundamentally different mechanism from renewal failure. Including them would pollute the model's signal.


In [0]:
billings['days_to_close'] = (billings['Closed_Date'] - billings['Prospect_Renewal_Date']).dt.days

# Apply churn labels
conditions = [
    # True churn: null closed date
    (billings['Prospect_Outcome'] == 'Churned') & (billings['Closed_Date'].isna()),
    # True churn: closed too late
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] > CHURN_DAYS_THRESHOLD),
    # Pre-churned: closed BEFORE prospect date
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] < 0),
    # On-time close within window  → also treat as churned if outcome='Churned' but closed exactly on day
    # Won
    billings['Prospect_Outcome'] == 'Won',
    # Open
    billings['Prospect_Outcome'] == 'Open',
]
choices = ['Churned', 'Churned', 'Pre_Churned', 'Won', 'Open']
billings['Churn_Label'] = np.select(conditions, choices, default='Churned')

# Summary
section("Churn Label Distribution")
summary = billings['Churn_Label'].value_counts().reset_index()
summary.columns = ['Label','Count']
summary['Pct'] = summary['Count'].apply(lambda x: pct(x, len(billings)))
display_df(summary, "All records by label")


## 4 · Validate Pre-Churned Exclusion

In [0]:
section("Pre-Churned Accounts — Validation")
pre_churn = billings[billings['Churn_Label'] == 'Pre_Churned'].copy()
print(f"Pre-churned accounts : {len(pre_churn):,}")
print(f"  → Closed BEFORE renewal date (days_to_close < 0)")
print(f"  → Min days diff : {pre_churn['days_to_close'].min():.0f}")
print(f"  → Max days diff : {pre_churn['days_to_close'].max():.0f}")
print(f"  → These are EXCLUDED from all analysis and model training.")

# Show a few examples
display_df(
    pre_churn[['Co_Ref','Prospect_Renewal_Date','Closed_Date','days_to_close','Prospect_Status']].head(10),
    "Sample pre-churned records"
)

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Pre-Churned vs True-Churned — Days-to-Close Distribution", fontsize=13)

true_churn = billings[billings['Churn_Label'] == 'Churned']
axes[0].hist(pre_churn['days_to_close'].dropna(), bins=40, color=PALETTE['amber'], edgecolor='#0a0e1a')
axes[0].set_title("Pre-Churned (days_to_close < 0)")
axes[0].set_xlabel("Days to Close")
axes[0].axvline(0, color=PALETTE['danger'], linestyle='--', linewidth=1.5, label="Prospect Date")
axes[0].legend()

axes[1].hist(true_churn['days_to_close'].dropna().clip(0, 200), bins=40, color=PALETTE['danger'], edgecolor='#0a0e1a')
axes[1].set_title("True Churned (null or >28 days)")
axes[1].set_xlabel("Days to Close (capped at 200)")
axes[1].axvline(CHURN_DAYS_THRESHOLD, color=PALETTE['amber'], linestyle='--', linewidth=1.5, label=f"{CHURN_DAYS_THRESHOLD}-day threshold")
axes[1].legend()

plt.tight_layout(); plt.show()


## 5 · Build Analysis DataFrame

In [0]:
# Keep Won + Churned only (model training population)
model_df = billings[billings['Churn_Label'].isin(['Won', 'Churned'])].copy()
model_df['Target']       = (model_df['Churn_Label'] == 'Churned').astype(int)
model_df['Renewal_Year'] = model_df['Prospect_Renewal_Date'].dt.year
model_df['Renewal_Month'] = model_df['Prospect_Renewal_Date'].dt.to_period('M').astype(str)
model_df['Renewal_MonthNum'] = model_df['Prospect_Renewal_Date'].dt.month

section("Final Analysis Population")
print(f"Total records (Won + Churned) : {len(model_df):,}")
print(f"  Won        : {(model_df['Target']==0).sum():,}")
print(f"  Churned    : {(model_df['Target']==1).sum():,}")
print(f"  Churn rate : {model_df['Target'].mean()*100:.2f}%")
print()

yr_summary = model_df.groupby('Renewal_Year').agg(
    Total=('Target','count'), Churned=('Target','sum')
).reset_index()
yr_summary['Churn_Rate%'] = yr_summary.apply(lambda r: pct(r['Churned'], r['Total']), axis=1)
display_df(yr_summary, "Yearly Churn Summary")

# ── Save to Delta for downstream notebooks ──────────────────
# model_df.to_spark().write.mode("overwrite").saveAsTable("churn_db.model_df")
# Alternatively, keep in memory if running all notebooks in one cluster session.
print("\n[NB02] model_df ready — shape:", model_df.shape)
